[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/radar/radar.ipynb)

# Radar Ranging

Radar ranging locates a moving reflector by timing light pulses sent out and received back. The same clock supplies both readings; their separation measures distance, and the changing spacing of the echoes reveals relative motion. Distances are expressed in light-travel units, so the speed of light is one. Reflection is instantaneous.

Projective spacetime algebra (STAP) extends Minkowski spacetime with a homogeneous coordinate. Here `x+t-w0` has one spatial and one time dimension; the zero-square `w0` supplies offsets and ideal directions. Boosts and translations then act on events, worldlines and clocks by the same motor sandwich. Each light leg is a linear map on homogeneous events, built from a join and an intersection; their composition maps emissions to receptions.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga@main"
%matplotlib inline

In [ ]:
import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.relativity.radar import render
from examples.util.animation import save_figure, show_animation
from examples.util.wgsl import image_figure

ga = Algebra("x+t-w0")
mv = NumpyContext(ga).multivector
# Events are intrinsic homogeneous points: a nonzero common scale leaves their location unchanged.
Event = ga.gatype.antivector()
# An extrinsic line equation pairs to zero with events on a worldline or simultaneous slice.
Slice = ga.gatype.vector()
Scalar = ga.gatype.scalar()
EventMap = ga.gatype((Event, Event))

# The signed blade complement gives an origin with homogeneous weight mv.w & origin == 1.
origin = mv.w.dual()
# Weight-zero events are ideal points: spatial and temporal directions.
across = mv.x.dual()
time = -mv.t.dual()
# Joining these directions to the origin gives light rays whose line squares are zero.
# Every ideal point itself squares to zero, including across and time.
outgoing = time + across
incoming = time - across

## Light pulses and echoes

A receding reflector stretches the intervals between returning pulses. In the spacetime diagram, time runs upward and light travels on the diagonals. The dark worldline belongs to the transmitting clock, the grey one to the reflector; orange paths carry outgoing light and blue paths carry its return. Numbers beside the clock mark emission and reception times.

In [ ]:
distance, velocity = 2, 0.25
n_pulses, pulse_spacing = 3, 0.75
end_margin = 0.5
n_frames, duration_ms = 99, 40

In [ ]:
def echo_maps(receiver: Slice, reflector: Slice, outward: Event, inward: Event,
              reference: Event) -> tuple[EventMap, EventMap]:
    """The reflection and echo maps for straight timelike worldlines and fixed light directions."""
    # Join an event to the outgoing ideal direction, then intersect the reflector's line equation.
    reflection = (Event & outward) ^ reflector
    returning = (Event & inward) ^ receiver
    # Fixed directions and straight worldlines make output weight a constant times input weight.
    # A unit-weight reference fixes that one scale for the whole map, preserving its linearity.
    reflection = reflection / (mv.w & reflection(reference))
    returning = returning / (mv.w & returning(reference))
    echo = returning(reflection)
    return reflection, echo


# xt squares to +1 and generates boosts; arctanh(velocity) is the reflector's rapidity.
# xw squares to zero and generates the spatial translation, applied after the boost.
placement = (mv.xw * (distance / 2)).exp() * (mv.xt * (np.arctanh(velocity) / 2)).exp()
target = placement >> mv.x
emitted = origin + time * (np.arange(n_pulses) * pulse_spacing)
reflection, echo = echo_maps(mv.x, target, outgoing, incoming, origin)
reflected, received = reflection(emitted), echo(emitted)

# Finite worldline segments bound the drawing after the last reception.
finish = (mv.t & received[-1]) + end_margin
target_end = target ^ (mv.t - mv.w * finish)
target_end = target_end / (mv.w & target_end)
worldlines = stack((stack((origin, origin + time * finish)),
                    stack((placement >> origin, target_end))))

show_animation(render.exchange_frames(worldlines, emitted, reflected, received, n_frames),
               "radar_echoes", duration_ms, n_frames=n_frames)

## Recovering the trajectory

The midpoint of the two clock readings assigns a time to the reflection; half their difference assigns a range. This locates the reflector when the pulse met it, earlier than the echo's arrival. Several echoes trace its motion without a clock on the reflector. In the diagram, green points mark the recovered events, and the dashed segment shows the range at the assigned time for the middle pulse.

In [ ]:
def reconstruct(emitted: Event, received: Event, clock: Slice, reference: Event,
                tangent: Event, spatial: Event) -> tuple[Event, Event, Scalar, Scalar]:
    """Locate reflections using an inertial observer's clock and the outgoing spatial direction."""
    radar_time = (clock & (received + emitted)) / 2
    radar_range = (clock & (received - emitted)) / 2
    simultaneous = reference + tangent * radar_time
    located = simultaneous + spatial * radar_range
    return located, simultaneous, radar_time, radar_range


def infer_motion(emitted: Event, received: Event, clock: Slice) -> tuple[Scalar, Scalar, Scalar]:
    """Infer echo stretch, relative velocity, and range at clock zero from two echoes."""
    # An inertial reflector makes the arrival time an affine function of emission time.
    stretch = (clock & (received[-1] - received[0])) / (clock & (emitted[-1] - emitted[0]))
    # Radar range changes with the difference of the two intervals; radar time with their sum.
    velocity = (stretch - 1) / (stretch + 1)
    first_time = (clock & (received[0] + emitted[0])) / 2
    first_range = (clock & (received[0] - emitted[0])) / 2
    initial_range = first_range - velocity * first_time
    return stretch, velocity, initial_range


located, simultaneous, radar_time, radar_range = reconstruct(
    emitted, received, mv.t, origin, time, across)
stretch, inferred_velocity, initial_range = infer_motion(emitted, received, mv.t)

figure = image_figure(render.reconstruction(worldlines, emitted, reflected, received,
                                          located, simultaneous, radar_time, radar_range, mv.t,
                                          render.EXCHANGE_WINDOW))
save_figure(figure, "radar_reconstruction")

In [ ]:
print("sent:", (mv.t & emitted).to_array())
print("returned:", (mv.t & received).to_array())
print("radar time:", radar_time.to_array())
print("range:", radar_range.to_array())
print("echo stretch:", stretch.to_array())
print("relative velocity (c):", inferred_velocity.to_array())
print("range at clock zero:", initial_range.to_array())

## Changing spacetime frame

A common boost and translation redraw the same experiment, including its transmitting clock. The reading `clock & event` stays unchanged when both are carried by the motor. The clock's line of simultaneous events tilts in the diagram, while its assigned time, range, and relative velocity retain their values.

In [ ]:
def frames(worldlines: Event, emitted: Event, reflected: Event, received: Event,
           n_frames: int):
    """A common boost about the middle of the experiment, followed by a translation."""
    final_rapidity, spatial_shift, time_shift = -0.65, 0.3, 0.2
    midpoint_time = (mv.t & (emitted[0] + received[-1])) / 2
    centre = (-mv.tw * (midpoint_time / 2)).exp()
    for fraction in np.linspace(0, 1, n_frames):
        boost = (mv.xt * (final_rapidity * fraction / 2)).exp()
        # The translation generator squares to zero, so its exponential is one plus the generator.
        translation = ((mv.xw * spatial_shift - mv.tw * time_shift) * (fraction / 2)).exp()
        # Conjugating the boost by centre anchors it at the midpoint event.
        motor = translation * centre * boost * centre.reverse()
        # The clock's unit and zero reading move with the experiment.
        clock = motor >> mv.t
        sent, returned = motor >> emitted, motor >> received
        located, simultaneous, radar_time, radar_range = reconstruct(
            sent, returned, clock, motor >> origin, motor >> time, motor >> across)
        yield motor >> worldlines, sent, motor >> reflected, returned, located, simultaneous, radar_time, radar_range, clock


show_animation((render.reconstruction(*state, render.FRAME_WINDOW)
                for state in frames(worldlines, emitted, reflected, received, n_frames)),
               "radar_frames", duration_ms, n_frames=n_frames, forward_then_back=True)

The midpoint convention is Einstein's radar assignment. Dolby and Gull's [*On Radar Time and the Twin ‘Paradox’*](https://arxiv.org/abs/gr-qc/0104077) develops it for observers whose motion also includes acceleration.